In [17]:
!pip install -q transformers==4.57.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 138.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [18]:
import torch
from transformers import AutoModel
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using:", device)

model = AutoModel.from_pretrained(
    "genbio-ai/genbio-pathfm",
    trust_remote_code=True
)

model = model.to(device)
model.eval()

Using: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/993 [00:00<?, ?B/s]

configuration_genbio_pathfm.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/genbio-ai/genbio-pathfm:
- configuration_genbio_pathfm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_genbio_pathfm.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/genbio-ai/genbio-pathfm:
- modeling_genbio_pathfm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/4.53G [00:00<?, ?B/s]

GenBioPathFMModel(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(1, 1536, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (rope_embed): RopePositionEmbedding()
    (blocks): ModuleList(
      (0-39): 40 x SelfAttentionBlock(
        (norm1): LayerNorm((1536,), eps=1e-06, elementwise_affine=True)
        (attn): SelfAttention(
          (qkv): Linear(in_features=1536, out_features=4608, bias=False)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1536, out_features=1536, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (norm2): LayerNorm((1536,), eps=1e-06, elementwise_affine=True)
        (mlp): SwiGLUFFN(
          (w1): Linear(in_features=1536, out_features=4096, bias=True)
          (w2): Linear(in_features=1536, out_features=4096, bias=True)
          (w3): Linear(in_features=4096, out_features=1536, bias=True)
      

In [19]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.697, 0.575, 0.728),
        std=(0.188, 0.240, 0.187),
    ),
])

In [20]:
from PIL import Image

image_path = "/content/TRIDENT/tests/HIST_files/TENX94_patch_00035.png"

image = Image.open(image_path).convert("RGB")

x = transform(image).unsqueeze(0).to(device)

print("Input shape:", x.shape)

with torch.inference_mode():
    embedding = model(x)

print("Embedding shape:", embedding.shape)

Input shape: torch.Size([1, 3, 224, 224])
Embedding shape: torch.Size([1, 4608])


In [21]:
print(embedding)

tensor([[-0.7285, -0.1346,  0.2831,  ..., -0.4297, -0.3189, -0.9732]],
       device='cuda:0')
